## Regression Evaluation: Retinal Age Gap

In [ ]:
%load_ext autoreload
%autoreload 2

# Run in parent dir
import os

os.chdir("../")

import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import nako
import numpy as np
import pandas as pd
import plot
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score

import models
import utils

plot.set_rc_params(kind='paper', notebook_dpi=120)

### Parameters

In [ ]:
method = 'finetune'
backbone = 'swin'
img_size= (224, 224)
normalize = 'none'
kfold = 1
experiment_name = f'reg_{backbone}_{img_size[0]}_{method}_{normalize}_{kfold}'

device = 'cuda:0'
feature_name = 'basis_age'
bins = list(range(20, 80, 5))
bins[0] = 19

# Where results are saved
project_dir = Path.cwd()
models_dir = project_dir.joinpath('checkpoints')
model_file = models_dir.joinpath(f'{experiment_name}.pt')
results_file = models_dir.joinpath(f'{experiment_name}.json')

plots_dir = project_dir.joinpath('plots')
plots_dir.mkdir(parents=True, exist_ok=True)

### Load Results

In [ ]:
with open(results_file, 'r') as f:
       results = json.load(f)

loss_train, loss_val, ids, types, targets, preds = [np.array(results[k]) for k in ['loss_train', 'loss_val', 'ID', 'type', 'targets', 'preds']]
del results

### Learning curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(6, 4))

ax.plot(range(loss_train.shape[0]), loss_train, '-o')
ax.plot(range(loss_val.shape[0]), loss_val, '-o')
ax.set_title('Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Average Loss')
ax.legend(['Train', 'Val'])

ax.set_ylim([0, 50])
plt.tight_layout()

### Load Dataset

In [ ]:
nako_paths = nako.get_nako_paths(dataset='590', image_res=str(img_size[0]))
transform = models.get_augmentations(img_size, normalization=nako.NORMALIZATION)['test']

dataset_train = nako.NakoDataset(nako_paths, nako.IMAGE_TYPES, transform=transform, split='train', kfold=kfold, feature_name=feature_name, drop_nan=True)
dataset_test = nako.NakoDataset(nako_paths, nako.IMAGE_TYPES, transform=transform, split='test', kfold=kfold, feature_name=feature_name, drop_nan=True)

### Age regression

In [ ]:
# Training dataset: to check distribution
df_train = pd.DataFrame({'ID': dataset_train.ids, 'targets': dataset_train.labels})
df_train = df_train.astype({'ID': int})

# Test dataset
df_test = pd.DataFrame({'ID': ids, 'type': types, 'targets': targets, 'preds': preds})
df_test = df_test.astype({'ID': int})
df_test['error_signed'] = df_test['preds'] - df_test['targets']
df_test['error'] = np.abs(df_test['error_signed'])
df_test['age_class'] = np.digitize(df_test['targets'], bins=bins) - 1
df_test['age_labels'] = df_test['age_class'].map(lambda x: bins[x])

#### Distribution of age values for train and test set

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 5), sharey=True)
ax = ax.flatten()

sns.histplot(df_train, x='targets', bins=bins, stat='percent', multiple='stack', edgecolor='black', ax=ax[0])
sns.histplot(df_test, x='targets', bins=bins, stat='percent', multiple='stack', edgecolor='black', ax=ax[1])

for i in range(ax.shape[0]):
    ax[i].set_xlabel('Age (5 year bins)')
    ax[i].xaxis.set_major_locator(ticker.MultipleLocator(bins[1] - bins[0]))
    ax[i].set_xlim([10, 100])

ax[0].set_ylabel('Percentage of participants')
ax[0].set_title('Age Distribution Training Set')
ax[1].set_title('Age Distribution Test Set')
plt.tight_layout()

#### Mean Absolute Error

In [ ]:
mae_test = mean_absolute_error(df_test['targets'], df_test['preds'])
print(f'Mean Absolute Error (test) = {mae_test:.3f}')

r2_test = r2_score(df_test['targets'], df_test['preds'])
print(f'R2 (test) = {r2_test:.3f}')

#### Distribution of error by age

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12, 5))
ax = ax.flatten()

df_test['targets_noise'] = df_test['targets'] + (0.5 * np.random.rand(df_test.shape[0]))
sns.scatterplot(x='targets_noise', y='preds', data=df_test, alpha=0.8, s=5, marker='.', linewidths=None, edgecolors='None', ax=ax[0])
sns.lineplot(x=[0, 100], y=[0, 100], color='grey', linewidth=0.8, legend=False, ax=ax[0])
ax[0].set_xlabel('Real Age')
ax[0].set_xlim([10, 90])
ax[0].set_ylim([10, 90])
# ax[0].grid()
# ax[0].set_aspect('equal', 'box')
ax[0].set_ylabel('Predicted Age')
ax[0].set_title('Predicted Age vs Real Age')

sns.boxplot(x='age_labels', y='error_signed', data=df_test, fill=False, ax=ax[1]) # showfliers=False
ax[1].set_ylim([-35, 35])
ax[1].axhspan(ax[1].get_ylim()[0], 0, *ax[1].get_xlim(), color='lightgray', alpha=0.5) # Add grey shading below y=0

ax[1].set_ylabel('Age Residuals')
ax[1].set_xlabel('Age (5 year bins)')
ax[1].set_title('Retinal Age Gap')
ax[1].legend(markerscale=1)   
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(1, 1)
plot.set_figsize(fig, 'col', height_ratio=0.8)

df_test['targets_noise'] = df_test['targets'] + (0.5 * np.random.rand(df_test.shape[0]))
sns.scatterplot(x='targets_noise', y='preds', data=df_test, alpha=0.8, s=5, marker='.', linewidths=None, edgecolors='None', ax=ax)
sns.lineplot(x=[0, 100], y=[0, 100], color='grey', linewidth=0.8, legend=False, ax=ax)
ax.set_xlim([11, 79])
ax.set_ylim([11, 79])
plot.dettach_axes(ax)
# plot.grid(ax)
# ax.set_aspect('equal', 'box')

ax.set_xlabel('Real Age [years]')
ax.set_ylabel('Predicted Age [years]')
# ax.set_title('Predicted Age vs Real Age')

plot.tight_layout()
fig.savefig('plots/age_prediction.svg')
fig.savefig('plots/age_prediction.png', bbox_inches='tight')

### Aggregate K-fold results on test set
The regression_val.py does the same on the validation set.

In [ ]:
experiment_names = [f'reg810_resnet50_224_finetune_none_{s}' for s in range(1, 6)]

mae, r2 = [], []
for experiment_name in experiment_names:
    results_file = models_dir.joinpath(f'{experiment_name}.json')
    with open(results_file, 'r') as f:
        results = json.load(f)

    targets, preds = [np.array(results[k]) for k in ['targets', 'preds']]
    del results

    mae_test = mean_absolute_error(targets, preds)
    r2_test = r2_score(targets, preds)

    mae.append(mae_test)
    r2.append(r2_test)

df_kfold = pd.DataFrame({'MAE': mae, 'R2': r2})
df_kfold

In [ ]:
df_kfold.describe()

### Compute performance estimates
Using bootstraping on the test set.

In [ ]:
experiment_names = ['reg_vit_224_finetune_none_4']

for experiment_name in experiment_names:
    print(experiment_name.split(sep='_')[1])
    results_file = models_dir.joinpath(f'{experiment_name}.json')

    with open(results_file, 'r') as f:
        results = json.load(f)

    targets, preds = [np.array(results[k]) for k in ['targets', 'preds']]

    utils.bootstrap_test(preds, targets, [mean_absolute_error, r2_score])